In [1]:
import json
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def load_jsonl(file_path):
    data = []
    with open(file_path, 'r') as f:
        for line in f:
            data.append(json.loads(line))
    return data

In [2]:
#model_name = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
model_name = "meta-llama/Llama-3.1-8B-Instruct"
output_file = "llama3.1_r1prompted_result.jsonl"
r1_file =  "results_all.jsonl"

In [3]:
data = load_jsonl(r1_file)
df = pd.DataFrame(data)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import os

def load_model(model_name, device="auto", cache_dir=None, dtype=torch.bfloat16):
    # Load tokenizer and model
    tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir=cache_dir)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=dtype,
        device_map=device,
        cache_dir=cache_dir
    )
    return model, tokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
cache_dir="/share/u/models"
model, tokenizer = load_model(os.path.join(cache_dir, model_name), device=device)

In [5]:
# Text Generation
if "Llama" in model_name:
    BOS = 128000
    USER = 128011
    ASSISTANT = 128012
    NEWLINE = 198
    THINK_START = 128013
    THINK_END = 128014
    EOS = 128001
elif "Qwen" in model_name:
    BOS = 151646
    USER = 151644
    ASSISTANT = 151645
    NEWLINE = 198
    THINK_START = 151648
    THINK_END = 151649
    EOS = 151643
else:
    raise ValueError(f"Unknown tokens for model {model_name}")

In [6]:
from rich.console import Console
from rich.panel import Panel
from rich.markdown import Markdown

def pprint(text):
    """Pretty print the model's generated text using rich."""
    console = Console(width=100)
    
    
    # Create markdown and display in a panel
    #md = Markdown(text.strip())
    console.print(Panel(text, border_style="blue"))


In [ ]:
idx = df['generated_text'][0].find("</think>")
pprint(df['generated_text'][0][:idx+len("</think>")])

In [8]:
from datasets import load_dataset

dataset = load_dataset("HuggingFaceH4/MATH-500")

if "R1" in model_name:
    def prompt_from_example(example, cot, tokenizer):
        # take everything up to the </think> part
        idx = cot['generated_text'].find("</think>")
        text = cot['generated_text'][:idx+len("</think>")]
        toks = tokenizer.encode(text, add_special_tokens=False)
        return toks, tokenizer.decode(toks, skip_special_tokens=False)
else:
    def prompt_from_example(example, cot, tokenizer):
        idx = cot['generated_text'].find("</think>")
        reasoning_text = cot['generated_text'][:idx+len("</think>")]
        idx = reasoning_text.find("<think>")
        reasoning_text = reasoning_text[idx:]
        idx = reasoning_text.find("\\boxed{")
        reasoning_text = reasoning_text[:idx]
        user_message = example['problem']
        math_suffix = " Please reason step by step within <think> </think> tags, and put your final answer as latex expression within \\boxed{}."
        messages = [
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": user_message + math_suffix},
        ]
        prompt = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
        prompt += reasoning_text + "\n</think>\n Answer:"
        toks = tokenizer.encode(prompt, add_special_tokens=False)
        return toks, prompt
    

In [9]:
toks, prompt = prompt_from_example(dataset['test'][0], df.iloc[0], tokenizer)
answer = dataset['test'][0]['answer']

In [ ]:
pprint(prompt)

In [ ]:
tokenizer

In [ ]:
# generate a continuation of prompt using huggingface text generation pipeline
from transformers import pipeline

if "R1" in model_name:
    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        do_sample=True,
        temperature=0.6,
        max_length=15000,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=EOS
    )
else:
    generator = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        do_sample=True,
        temperature=0.6,
        max_length=15000,
        pad_token_id=128002,
        eos_token_id=tokenizer.eos_token_id,
        max_new_tokens=1000
    )

In [ ]:
out = generator(prompt)
pprint(out[0]['generated_text'])

In [ ]:
import re
from math500.grader import grade_answer

if "R1" in model_name:
    def parse_answer(generated_text):
        matches = re.search("</think>", generated_text)
        if matches is None:
            return ""
        generated_answer = generated_text[matches.end():]
        # in generated answer select the content of \boxed{}
        matches = re.search("\\\\boxed{", generated_answer)
        if matches is None:
            return ""
        generated_answer = generated_answer[matches.end():]
        # search all the way to the end of the string   }
        reversed = generated_answer[::-1]
        matches = re.search("}", reversed)
        if matches is None:
            return ""
        generated_answer = generated_answer[:len(generated_answer) - matches.start()]
        return generated_answer
else:
    def parse_answer(generated_text):
        end_idx = generated_text.find("<|start_header_id|>assistant<|end_header_id|>")
        generated_answer = generated_text[end_idx:]
        idx = generated_answer.find("</think>")
        generated_answer = generated_answer[idx+len("</think>"):]
        matches = re.search("\\\\boxed{", generated_answer)
        if matches is None:
            return ""
        generated_answer = generated_answer[matches.end():]
        # search all the way to the end of the string   }
        reversed = generated_answer[::-1]
        matches = re.search("}", reversed)
        if matches is None:
            return ""
        generated_answer = generated_answer[:len(generated_answer) - matches.start()]
        return generated_answer
parsed_answer = parse_answer(out[0]['generated_text'])
print(parsed_answer)
print(parsed_answer, answer)
grade_answer(parsed_answer, answer)

In [ ]:
# evaluate question by question and store results in both dataframe and jsonl file
import pandas as pd
import json
from tqdm import tqdm

df2 = pd.DataFrame(columns=['problem', 'answer', 'generated_answer', 'correct'])
n_correct = 0
for idx, example in tqdm(enumerate(dataset['test'])):
    try: 
        toks, prompt = prompt_from_example(example, df.iloc[idx], tokenizer)
        out = generator(prompt)
        generated_answer = parse_answer(out[0]['generated_text'])
        example['parsed_answer'] = generated_answer
        example['correct'] = grade_answer(generated_answer, example['answer'])
        example['generated_text'] = out[0]['generated_text']
        n_correct += int(example['correct'])
        print(n_correct / (idx + 1))

        # add to both dataframe and jsonl file
        df2 = pd.concat([df2, pd.DataFrame([example])], ignore_index=True)
        
        # Write to jsonl file immediately after each example
        with open(output_file, 'a') as f:
            f.write(json.dumps(example) + '\n')
            f.flush() # Ensure it's written to disk
            
        #if idx > 3:
        #    break
    except Exception as e:
        print(e)
        print(f"Error with example {idx}")
        with open(output_file, 'a') as f:
            f.write(json.dumps(example) + '\n')
            f.flush()
        continue